In [59]:
training_text = "Byte Pair Encoding is a simple tokenization algorithm. It starts from raw bytes and repeatedly merges the most frequent pair. Tokenizers like GPT-2's use byte-level BPE — even emoji 😀 and accented characters like café work, because everything is just bytes underneath."

# str.encode("utf-8") gives a bytes object; wrapping it in list() gives
# the raw byte values as integers in the range 0..255
tokens = list(training_text.encode("utf-8"))

print(f"characters: {len(training_text)}")
print(f"bytes:      {len(tokens)}")
print(tokens)

characters: 268
bytes:      274
[66, 121, 116, 101, 32, 80, 97, 105, 114, 32, 69, 110, 99, 111, 100, 105, 110, 103, 32, 105, 115, 32, 97, 32, 115, 105, 109, 112, 108, 101, 32, 116, 111, 107, 101, 110, 105, 122, 97, 116, 105, 111, 110, 32, 97, 108, 103, 111, 114, 105, 116, 104, 109, 46, 32, 73, 116, 32, 115, 116, 97, 114, 116, 115, 32, 102, 114, 111, 109, 32, 114, 97, 119, 32, 98, 121, 116, 101, 115, 32, 97, 110, 100, 32, 114, 101, 112, 101, 97, 116, 101, 100, 108, 121, 32, 109, 101, 114, 103, 101, 115, 32, 116, 104, 101, 32, 109, 111, 115, 116, 32, 102, 114, 101, 113, 117, 101, 110, 116, 32, 112, 97, 105, 114, 46, 32, 84, 111, 107, 101, 110, 105, 122, 101, 114, 115, 32, 108, 105, 107, 101, 32, 71, 80, 84, 45, 50, 39, 115, 32, 117, 115, 101, 32, 98, 121, 116, 101, 45, 108, 101, 118, 101, 108, 32, 66, 80, 69, 32, 226, 128, 148, 32, 101, 118, 101, 110, 32, 101, 109, 111, 106, 105, 32, 240, 159, 152, 128, 32, 97, 110, 100, 32, 97, 99, 99, 101, 110, 116, 101, 100, 32, 99, 104, 97, 114, 97, 

In [60]:
def bytes_to_unicode():
    """GPT-2's reversible map: every byte 0..255 -> its own distinct printable char.

    Bytes that are already printable ASCII/Latin-1 map to themselves; the rest
    (space, control chars, etc.) get fresh code points starting at 256, so nothing
    ever renders as a blank or a weird glyph. Byte 32 (space) becomes 'Ġ'.
    """
    bs = list(range(ord("!"), ord("~") + 1)) + \
         list(range(ord("¡"), ord("¬") + 1)) + \
         list(range(ord("®"), ord("ÿ") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:            # byte isn't already a printable char
            bs.append(b)
            cs.append(256 + n)     # hand it a fresh, guaranteed-printable code point
            n += 1
    return {b: chr(c) for b, c in zip(bs, cs)}

byte_encoder = bytes_to_unicode()   # byte value (0..255) -> visible char

def render_piece(idx, vocab):
    """Visible form of a token: expand it to its bytes, map each through byte_encoder."""
    return "".join(byte_encoder[b] for b in vocab[idx])

def get_pair_counts(tokens, counts=None):
    """Count adjacent pairs in `tokens`.

    Pass an existing `counts` dict to keep accumulating into it — that lets us
    tally pairs across many chunks without them bleeding into one another.
    """
    counts = {} if counts is None else counts
    for pair in zip(tokens, tokens[1:]):  # iterate over consecutive pairs
        counts[pair] = counts.get(pair, 0) + 1
    return counts

pair_counts = get_pair_counts(tokens)

# show the 10 most frequent pairs, with the bytes shown via the GPT-2 table
for pair, count in sorted(pair_counts.items(), key=lambda kv: kv[1], reverse=True)[:10]:
    print(f"{pair}  ->  {count}   {byte_encoder[pair[0]]}{byte_encoder[pair[1]]}")

(115, 32)  ->  9   sĠ
(116, 101)  ->  7   te
(101, 32)  ->  7   eĠ
(121, 116)  ->  5   yt
(32, 97)  ->  5   Ġa
(101, 110)  ->  5   en
(101, 114)  ->  5   er
(107, 101)  ->  4   ke
(116, 104)  ->  4   th
(116, 32)  ->  4   tĠ


In [61]:
def merge(tokens, pair, new_id):
    """Replace every occurrence of `pair` in `tokens` with the single token `new_id`."""
    merged = []
    i = 0
    while i < len(tokens):
        # if this position starts the pair, emit new_id and skip both elements
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            merged.append(new_id)
            i += 2
        else:
            merged.append(tokens[i])
            i += 1
    return merged

# find the most frequent pair and merge it into token 256
top_pair = max(pair_counts, key=pair_counts.get)
tokens2 = merge(tokens, top_pair, 256)

print(f"merging {top_pair} ({byte_encoder[top_pair[0]]}{byte_encoder[top_pair[1]]}) -> 256")
print(f"length before: {len(tokens)}")
print(f"length after:  {len(tokens2)}")

merging (115, 32) (sĠ) -> 256
length before: 274
length after:  265


In [62]:
print(list(tokens2))

[66, 121, 116, 101, 32, 80, 97, 105, 114, 32, 69, 110, 99, 111, 100, 105, 110, 103, 32, 105, 256, 97, 32, 115, 105, 109, 112, 108, 101, 32, 116, 111, 107, 101, 110, 105, 122, 97, 116, 105, 111, 110, 32, 97, 108, 103, 111, 114, 105, 116, 104, 109, 46, 32, 73, 116, 32, 115, 116, 97, 114, 116, 256, 102, 114, 111, 109, 32, 114, 97, 119, 32, 98, 121, 116, 101, 256, 97, 110, 100, 32, 114, 101, 112, 101, 97, 116, 101, 100, 108, 121, 32, 109, 101, 114, 103, 101, 256, 116, 104, 101, 32, 109, 111, 115, 116, 32, 102, 114, 101, 113, 117, 101, 110, 116, 32, 112, 97, 105, 114, 46, 32, 84, 111, 107, 101, 110, 105, 122, 101, 114, 256, 108, 105, 107, 101, 32, 71, 80, 84, 45, 50, 39, 256, 117, 115, 101, 32, 98, 121, 116, 101, 45, 108, 101, 118, 101, 108, 32, 66, 80, 69, 32, 226, 128, 148, 32, 101, 118, 101, 110, 32, 101, 109, 111, 106, 105, 32, 240, 159, 152, 128, 32, 97, 110, 100, 32, 97, 99, 99, 101, 110, 116, 101, 100, 32, 99, 104, 97, 114, 97, 99, 116, 101, 114, 256, 108, 105, 107, 101, 32, 99, 97, 

In [63]:
import regex as re     # the 'regex' package (not stdlib 're'); pip install regex

# GPT-2's / GPT-4's pre-tokenization pattern. Applied BEFORE BPE, it slices text
# into word-ish chunks. BPE then runs *inside* each chunk, so a merge can never
# span two chunks — no more merges leaking across word/space boundaries.
#   '(?:[sdmt]|ll|ve|re)  contractions: 's 't 'll 've ...
#    ?\p{L}+              a run of letters, with an optional leading space
#    ?\p{N}+              a run of digits, with an optional leading space
#    ?[^\s\p{L}\p{N}]+    a run of punctuation/symbols
#   \s+(?!\S) | \s+       trailing / interior whitespace
SPLIT_PATTERN = re.compile(
    r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)

chunks = re.findall(SPLIT_PATTERN, training_text)
print(f"{len(chunks)} chunks")
print(chunks[:14])

51 chunks
['Byte', ' Pair', ' Encoding', ' is', ' a', ' simple', ' tokenization', ' algorithm', '.', ' It', ' starts', ' from', ' raw', ' bytes']


In [66]:
# ---- hyperparameters ----
vocab_size = 276                    # target vocabulary size (must be >= 256)
num_merges = vocab_size - 256       # merges to learn on top of the 256 base bytes

# pre-tokenize: each chunk becomes its own little list of bytes.
# merges happen WITHIN a chunk, never across the boundary between two.
chunk_ids = [list(ch.encode("utf-8")) for ch in re.findall(SPLIT_PATTERN, training_text)]

merges = {}                                    # (int, int) -> new_id   <- the learned model
vocab = {i: bytes([i]) for i in range(256)}    # id -> the raw bytes it expands to

for k in range(num_merges):
    counts = {}
    for chunk in chunk_ids:
        get_pair_counts(chunk, counts)         # accumulate across all chunks
    if not counts:
        break                                  # nothing left to merge anywhere
    top_pair = max(counts, key=counts.get)     # most frequent pair, corpus-wide
    new_id = 256 + k

    chunk_ids = [merge(chunk, top_pair, new_id) for chunk in chunk_ids]
    merges[top_pair] = new_id
    vocab[new_id] = vocab[top_pair[0]] + vocab[top_pair[1]]

    print(f"merge {k + 1:2d}: {top_pair} -> {new_id}   "
          f"({counts[top_pair]}x)   {render_piece(new_id, vocab)!r}")

# special tokens get reserved ids ABOVE every byte and merge id
special_tokens = {"<|endoftext|>": vocab_size}

ids = [i for chunk in chunk_ids for i in chunk]   # flattened, for reporting/checks
print()
print(f"tokens: {len(tokens)} -> {len(ids)}")
print(f"vocab size:  {len(vocab)} (+{len(special_tokens)} special)")
print(f"compression: {len(tokens) / len(ids):.2f}x")

merge  1: (116, 101) -> 256   (7x)   'te'
merge  2: (32, 97) -> 257   (5x)   'Ġa'
merge  3: (101, 110) -> 258   (5x)   'en'
merge  4: (121, 256) -> 259   (4x)   'yte'
merge  5: (116, 104) -> 260   (4x)   'th'
merge  6: (32, 98) -> 261   (4x)   'Ġb'
merge  7: (101, 114) -> 262   (4x)   'er'
merge  8: (115, 116) -> 263   (3x)   'st'
merge  9: (261, 259) -> 264   (3x)   'Ġbyte'
merge 10: (110, 100) -> 265   (3x)   'nd'
merge 11: (101, 118) -> 266   (3x)   'ev'
merge 12: (97, 105) -> 267   (2x)   'ai'
merge 13: (267, 114) -> 268   (2x)   'air'
merge 14: (105, 110) -> 269   (2x)   'in'
merge 15: (269, 103) -> 270   (2x)   'ing'
merge 16: (32, 105) -> 271   (2x)   'Ġi'
merge 17: (271, 115) -> 272   (2x)   'Ġis'
merge 18: (111, 107) -> 273   (2x)   'ok'
merge 19: (273, 258) -> 274   (2x)   'oken'
merge 20: (274, 105) -> 275   (2x)   'okeni'

tokens: 274 -> 211
vocab size:  276 (+1 special)
compression: 1.30x


In [67]:
def save(path, merges, pattern, special_tokens):
    """Write the model: regex pattern, special tokens, then merges (in learned order).

    Merges still store only the *pairs* — the id each produces is implicit from
    line order. The pattern and special tokens are what let a fresh load encode
    text exactly the way this model was trained to.
    """
    with open(path, "w", encoding="utf-8") as f:
        f.write("bpe v2\n")                     # bumped version: now carries more
        f.write(pattern + "\n")                 # the regex, on one line
        f.write(f"{len(special_tokens)}\n")
        for tok, idx in special_tokens.items():
            f.write(f"{tok} {idx}\n")           # token names have no spaces
        f.write(f"{len(merges)}\n")
        for (a, b) in merges:
            f.write(f"{a} {b}\n")

save("tokenizer.model", merges, SPLIT_PATTERN.pattern, special_tokens)

# show what we just wrote
with open("tokenizer.model", encoding="utf-8") as f:
    print(f.read())

bpe v2
'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+
1
<|endoftext|> 276
20
116 101
32 97
101 110
121 256
116 104
32 98
101 114
115 116
261 259
110 100
101 118
97 105
267 114
105 110
269 103
32 105
271 115
111 107
273 258
274 105



In [68]:
def load(path):
    """Rebuild everything the model needs: merges, vocab, pattern, special tokens."""
    merges = {}
    vocab = {i: bytes([i]) for i in range(256)}   # base bytes, same as training
    special_tokens = {}
    with open(path, encoding="utf-8") as f:
        assert f.readline().strip() == "bpe v2"   # check the header
        pattern = f.readline().rstrip("\n")       # keep internal spaces, drop newline
        n_special = int(f.readline())
        for _ in range(n_special):
            tok, idx = f.readline().rstrip("\n").rsplit(" ", 1)
            special_tokens[tok] = int(idx)
        n_merges = int(f.readline())
        for k in range(n_merges):
            a, b = map(int, f.readline().split())
            new_id = 256 + k                       # id is implicit from line order
            merges[(a, b)] = new_id
            vocab[new_id] = vocab[a] + vocab[b]
    return merges, vocab, pattern, special_tokens

loaded_merges, loaded_vocab, loaded_pattern, loaded_special = load("tokenizer.model")

# confirm the round-trip is exact
print("merges match: ", loaded_merges == merges)
print("vocab match:  ", loaded_vocab == vocab)
print("pattern match:", loaded_pattern == SPLIT_PATTERN.pattern)
print("special match:", loaded_special == special_tokens)

merges match:  True
vocab match:   True
pattern match: True
special match: True


In [69]:
def decode(ids, vocab, special_tokens=None):
    """ids -> text. Normal ids expand to bytes; special ids emit their literal string."""
    inverse_special = {idx: tok for tok, idx in (special_tokens or {}).items()}
    out = []       # finished string pieces, in order
    buf = []       # run of ordinary byte-tokens waiting to be decoded together
    for idx in ids:
        if idx in inverse_special:
            if buf:                                        # flush pending bytes first
                out.append(b"".join(buf).decode("utf-8", errors="replace"))
                buf = []
            out.append(inverse_special[idx])              # then the special token text
        else:
            buf.append(vocab[idx])
    if buf:                                                # flush whatever's left
        out.append(b"".join(buf).decode("utf-8", errors="replace"))
    return "".join(out)

print(decode(ids, vocab)[:60])                            # round-tripped training text
print(decode([72, 105, 276], vocab, special_tokens))      # 'Hi' + <|endoftext|>

Byte Pair Encoding is a simple tokenization algorithm. It st
Hi<|endoftext|>


In [70]:
def _encode_chunk(text, merges):
    """Byte-level BPE on a single pre-token chunk: apply merges in rank order."""
    ids = list(text.encode("utf-8"))
    while len(ids) >= 2:
        counts = get_pair_counts(ids)
        # among pairs present, pick the one learned earliest (lowest rank).
        # pairs we never learned get inf, so they're never chosen.
        pair = min(counts, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break                       # nothing left that we know how to merge
        ids = merge(ids, pair, merges[pair])
    return ids

def encode(text, merges, special_tokens=None):
    """text -> ids. Peel off special tokens, pre-tokenize the rest, BPE each chunk."""
    special_tokens = special_tokens or {}
    if special_tokens:
        # split on the special tokens but KEEP them (capturing group)
        specials_re = "(" + "|".join(re.escape(s) for s in special_tokens) + ")"
        parts = re.split(specials_re, text)
    else:
        parts = [text]

    ids = []
    for part in parts:
        if part in special_tokens:
            ids.append(special_tokens[part])            # emit its reserved id directly
        else:
            for chunk in re.findall(SPLIT_PATTERN, part):   # pre-tokenize
                ids.extend(_encode_chunk(chunk, merges))    # then BPE within the chunk
    return ids

print(encode("bytes ", merges))
print(encode("say bytes <|endoftext|> and more", merges, special_tokens))

[98, 259, 115, 32]
[115, 97, 121, 264, 115, 32, 276, 257, 265, 32, 109, 111, 114, 101]


In [79]:
# --- checks ---

# 1. encoding the training text reproduces exactly the ids the trainer produced
print("encode matches training ids:", encode(training_text, merges) == ids)

# 2. decode(encode(x)) == x  for training text and for unseen text
print("round-trip (training):", decode(encode(training_text, merges), vocab) == training_text)

unseen = "café bytes and emoji 😀 underneath!"
print("round-trip (unseen):  ", decode(encode(unseen, merges), vocab) == unseen)

# 3. special tokens survive a round-trip and get their reserved id
special_text = "hello <|endoftext|> world"
enc = encode(special_text, merges, special_tokens)
print("special id present:   ", special_tokens["<|endoftext|>"] in enc)
print("round-trip (special): ", decode(enc, vocab, special_tokens) == special_text)

# 4. everything works with the model reloaded from disk
print("round-trip (loaded):  ",
      decode(encode(special_text, loaded_merges, loaded_special),
             loaded_vocab, loaded_special) == special_text)

encode matches training ids: True
round-trip (training): True
round-trip (unseen):   True
special id present:    True
round-trip (special):  True
round-trip (loaded):   True


In [82]:
print(encode("Hello, My name is Aksh <|endoftext|>", merges, special_tokens))

[72, 101, 108, 108, 111, 44, 32, 77, 121, 32, 110, 97, 109, 101, 272, 32, 65, 107, 115, 104, 32, 276]


In [83]:
print(special_tokens)

{'<|endoftext|>': 276}


In [81]:
print(decode([276],vocab, special_tokens))

<|endoftext|>
